In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy import stats

def get_conf_int(alpha, lr, X, y):
    
    """
    Returns (1-alpha) 2-sided confidence intervals
    for sklearn.LinearRegression coefficients
    as a pandas DataFrame
    """
    X_aux = X.copy()
    X_aux.insert(0, 'const', 1)
    dof = -np.diff(X_aux.shape)[0]
    mse = (np.sum((y - pd.DataFrame(lr.predict(X))) ** 2) / dof).to_list()[0]
    var_params = np.diag(np.linalg.inv(X_aux.T.dot(X_aux)))
    t_val = stats.t.isf(alpha/2, dof)
    gap = t_val * np.sqrt(mse * var_params)

    return gap

In [ ]:
pe_tube_orig = pd.read_csv('pe_tube.csv', parse_dates=['Deployment Date', 'Recovery Date'])
for source in ["JS_DU", "JS_ME"]:
    pe_tube = pe_tube_orig.loc[pe_tube_orig['Source'] == source]
    pe_tube['Deployment Time'] = pd.to_timedelta(pe_tube['Deployment Time'])
    pe_tube['Functional Group'].replace({
        "CO2H": "PFCA", "SO3H": "PFSA",

    }, inplace=True)

    pe_tube['Concentration from model'] = 1e3 * pe_tube['Concentration in nanogram per sampler'] / (
    pe_tube['Deployment Time'].dt.days * pe_tube['Sampling Rate Model [mL/day]']
    )
    pe_tube['Concentration uncertainty from model'] = 1e3 * pe_tube['Std Sampling Rate Model [mL/day]'] *\
    pe_tube['Concentration in nanogram per sampler'] / (
    pe_tube['Deployment Time'].dt.days * pe_tube['Sampling Rate Model [mL/day]']**2
    )
    model = LinearRegression(fit_intercept=False)
    model.fit(
        pe_tube['Grab Sample Average'].values.reshape(-1, 1),
        pe_tube['Concentration from model'].values.reshape(-1, 1)
    )
    y_fit = model.predict(pe_tube['Grab Sample Average'].values.reshape(-1, 1))
    gap = get_conf_int(
        0.32, model, X=pd.DataFrame(pe_tube['Grab Sample Average'].values.reshape(-1, 1)),
        y=pd.DataFrame(pe_tube['Concentration from model'].values.reshape(-1, 1)))

    r2 = np.round(model.score(y_fit, pe_tube['Concentration from model'].values.reshape(-1, 1)), 2)
    slope = np.round(model.coef_[0], 3)[0]
    slope_std = np.round(gap[1], 3)

    pe_tube['Percentage of Grab'] = 100 *pe_tube['Concentration from model'] / pe_tube['Grab Sample Average']

    avg_percentage = np.round(pe_tube['Percentage of Grab'].mean(), 1)

    fig, ax = plt.subplots(1, 1)
    for group in ["PFCA", "PFSA"]:
        group_data = pe_tube.loc[pe_tube['Functional Group'] == group]
        if group == "PFCA":
            plotcolor = 'purple'
        elif group == "PFSA":
            plotcolor = 'green'
        ax.errorbar(
            x=group_data['Grab Sample Average'], xerr=group_data['Grab Sample Standard Deviation'],
            y=group_data['Concentration from model'], yerr=group_data['Concentration uncertainty from model'],
            label=group, alpha=0.7, marker='^', color=plotcolor, ecolor='grey', linestyle='None',
            )
    ax.plot(
        pe_tube['Grab Sample Average'], y_fit, color='cyan', label='Linear Fit', linestyle='dotted'
        )
    #ax.set_xlim(0, 10)
    #ax.set_ylim(0, 10)
    ax.set_xlabel('Grab Measured [ng/L]')
    ax.set_ylabel('Sampler Measured [ng/L]')
    ax.legend()
    ax.set_title(f"Regression Slope: {slope} ± {slope_std}, R2: {r2}, \n Average percentage of grab: {avg_percentage} %")
    if source == "JS_DU":
        ax.plot(
            [0, 350], [0, 350], color='black', label='1:1 line', linestyle='--'
            )
        fig.savefig('plots/duluth_challenge.png')
    elif source == "JS_ME":
        ax.plot(
            [0, 50], [0, 50], color='black', label='1:1 line', linestyle='--'
            )
        fig.savefig('plots/maine_challenge.png')
